# Facility location: the decision before the transportation problem

The transportation notebook took the warehouses as given and asked what to ship from each. This one
asks the question that comes first: **which warehouses to open at all.** Each candidate site has a
fixed cost to open, a capacity, and a per-unit cost to run. Five customers need serving. Open too
many and you pay fixed costs for nothing; open too few and you ship across the map.

One yes/no per site is all that changes, and it changes everything about how the problem is solved.
A linear program cannot say "yes or no" — it says "0.62", opens 62% of a warehouse, and pays 62% of
the fixed cost. That answer is meaningless and it is also **cheaper than any real one**, which is
exactly what makes it useful. This notebook builds the integer model, then the relaxation that
under-prices it, then does by hand the thing the solver does to close the gap between them.

## Licence setup

In [1]:
import gurobipy as gp

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)        # start silent: the licence banner, and its licence number, stay out of the outputs
try:
    from google.colab import userdata
    try:
        env.setParam("WLSACCESSID", userdata.get("GRB_WLSACCESSID"))
        env.setParam("WLSSECRET",   userdata.get("GRB_WLSSECRET"))
        env.setParam("LICENSEID",   int(userdata.get("GRB_LICENSEID")))
    except userdata.SecretNotFoundError:
        raise SystemExit("Add GRB_WLSACCESSID, GRB_WLSSECRET and GRB_LICENSEID as Colab Secrets "
                         "(key icon, left sidebar), then re-run this cell.")
    env.start()
    print("licence: Colab Secrets (WLS)")
except ImportError:
    env.start()
    print("licence: local gurobi.lic")

licence: local gurobi.lic


## Three tables

Warehouses (capacity, fixed cost, variable cost), customers (demand), and a shipping cost for every
pair. These arrived with the original notebook as three CSV files — instance data, indexed by the
model's sets, already living where Part 4 of the standard says it should.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "src")))
from orteach import tolerance
from orteach.facility_location import load_ufl

inst = load_ufl()

print(f"{'warehouse':>10} {'capacity':>9} {'fixed':>7} {'variable':>9}")
for w in inst.warehouses:
    print(f"{w:>10} {inst.max_capacity[w]:9.0f} {inst.fixed_cost[w]:7.0f} {inst.variable_cost[w]:9.0f}")
print()
print("demand:", inst.demand, "   total", sum(inst.demand.values()))
print()
print(f"{'ship cost':>10}" + "".join(f"{'cust ' + c:>8}" for c in inst.customers))
for w in inst.warehouses:
    print(f"{'wh ' + w:>10}" + "".join(f"{inst.shipping[(w, c)]:8.0f}" for c in inst.customers))

 warehouse  capacity   fixed  variable
         1        80    1000        20
         2        80    1500        17
         3        80    1700        13
         4        80    1400        25
         5        80    1200        33

demand: {'1': 30.0, '2': 40.0, '3': 50.0, '4': 35.0, '5': 40.0}    total 195.0

 ship cost  cust 1  cust 2  cust 3  cust 4  cust 5
      wh 1       8      21      42      12      37
      wh 2      21      10      31      24      40
      wh 3      42      31       4      14      32
      wh 4      12      24      14       7      12
      wh 5      37      40      32      12      10


Total demand is 195 and every warehouse holds 80, so at least three must open. Beyond that, nothing is
obvious: the cheapest site to open is the dearest to run, the cheapest to run is the second dearest to
open, and the shipping matrix has a warehouse that is close to almost everybody.

## Predict before building anything

**Write down which three warehouses you would open, and roughly what the total would be.** Then
predict something harder: if you let the model open *fractions* of a warehouse, does the cost go down
a little or a lot?

## The model, one piece at a time

Three kinds of decision. What to ship along each pair — continuous. Whether to open each site —
**binary**, the one variable in this library so far that is genuinely yes/no. And how much capacity
to stand up at each site — continuous, because the original formulation kept it as its own number, and
"how much did we build" is a number a reader wants to point at.

In [3]:
m = gp.Model(env=env)
tolerance.apply(m)
m.ModelSense = gp.GRB.MINIMIZE

ship = m.addVars(inst.shipping.keys(), lb=0.0, name="ship")
opened = m.addVars(inst.warehouses, vtype=gp.GRB.BINARY, name="open")
prov = m.addVars(inst.warehouses, lb=0.0, name="provisioned")
m.update()
print(f"{m.NumVars} variables: {len(ship)} ship, {len(opened)} open (binary), {len(prov)} provisioned")

35 variables: 25 ship, 5 open (binary), 5 provisioned


Three costs, three sums. Shipping is per unit moved; fixed is per site opened; variable is per unit of
capacity stood up.

In [4]:
m.setObjective(gp.quicksum(inst.shipping[a] * ship[a] for a in inst.shipping)
               + gp.quicksum(inst.fixed_cost[w] * opened[w] for w in inst.warehouses)
               + gp.quicksum(inst.variable_cost[w] * prov[w] for w in inst.warehouses))
print("objective has", m.getObjective().size(), "terms")

objective has 0 terms


Every customer gets what it needs.

In [5]:
m.addConstrs((ship.sum("*", c) >= inst.demand[c] for c in inst.customers), name="demand")
m.update()
print(f"{m.NumConstrs} demand rows")

5 demand rows


Two coupling rows, and **the second is the whole problem.** A warehouse ships no more than it has
stood up. And it stands up no more than its capacity — *times whether it is open.* If `open[w]` is 0
the right-hand side is 0 and the site is dark. This is the line that links a yes/no to everything
downstream of it.

In [6]:
m.addConstrs((ship.sum(w, "*") <= prov[w] for w in inst.warehouses), name="capacity_used")
m.addConstrs((prov[w] <= inst.max_capacity[w] * opened[w] for w in inst.warehouses), name="capacity_built")
m.update()
print(f"{m.NumConstrs} constraints in total")
assert m.NumConstrs == len(inst.customers) + 2 * len(inst.warehouses)

15 constraints in total


**Predict before solving.** Three sites, you said. Which three, and does the answer use every site it
opens to capacity?

In [7]:
m.optimize()

mip_obj = m.ObjVal
mip_open = {w: opened[w].X for w in inst.warehouses}
mip_ship = {a: ship[a].X for a in inst.shipping}
mip_prov = {w: prov[w].X for w in inst.warehouses}

print(f"cheapest total: {mip_obj:,.0f}\n")
for w in inst.warehouses:
    if mip_open[w] > 0.5:
        out = sum(q for (ww, c), q in mip_ship.items() if ww == w)
        print(f"  warehouse {w}: open, ships {out:5.0f} of {inst.max_capacity[w]:.0f}")
print(f"\nsites opened: {[w for w, v in mip_open.items() if v > 0.5]}")

cheapest total: 9,880

  warehouse 1: open, ships    75 of 80
  warehouse 3: open, ships    80 of 80
  warehouse 4: open, ships    40 of 80

sites opened: ['1', '3', '4']


Split the cost three ways to see where the money went.

In [8]:
fixed    = sum(inst.fixed_cost[w] * mip_open[w] for w in inst.warehouses)
variable = sum(inst.variable_cost[w] * mip_prov[w] for w in inst.warehouses)
shipping = sum(inst.shipping[a] * q for a, q in mip_ship.items())
print(f"fixed    {fixed:8,.0f}")
print(f"variable {variable:8,.0f}")
print(f"shipping {shipping:8,.0f}")
print(f"total    {fixed + variable + shipping:8,.0f}")

fixed       4,100
variable    3,540
shipping    2,240
total       9,880


Notice `provisioned`. It never comes back larger than what actually ships, because provisioning costs
money and buys nothing beyond the shipments it enables. The third variable is not wrong — it is
redundant, and knowing *which* variables in a model are redundant is a skill.

In [9]:
for w in inst.warehouses:
    out = sum(q for (ww, c), q in mip_ship.items() if ww == w)
    print(f"  warehouse {w}: provisioned {mip_prov[w]:5.0f}   shipped {out:5.0f}")

  warehouse 1: provisioned    75   shipped    75
  warehouse 2: provisioned     0   shipped     0
  warehouse 3: provisioned    80   shipped    80
  warehouse 4: provisioned    40   shipped    40
  warehouse 5: provisioned     0   shipped     0


## Now drop the one thing that made it hard

Same model, but `open` is continuous in [0, 1]. This is the **LP relaxation**, and the question is
what a solver does with the freedom to open sixty per cent of a building.

**Predict:** how many sites will be partially open, and is the total closer to the integer answer
above or well below it?

In [10]:
m_lp = gp.Model(env=env)
tolerance.apply(m_lp)
m_lp.ModelSense = gp.GRB.MINIMIZE
ship_lp = m_lp.addVars(inst.shipping.keys(), lb=0.0, name="ship")
open_lp = m_lp.addVars(inst.warehouses, lb=0.0, ub=1.0, name="open")      # continuous now
prov_lp = m_lp.addVars(inst.warehouses, lb=0.0, name="provisioned")
m_lp.setObjective(gp.quicksum(inst.shipping[a] * ship_lp[a] for a in inst.shipping)
                  + gp.quicksum(inst.fixed_cost[w] * open_lp[w] for w in inst.warehouses)
                  + gp.quicksum(inst.variable_cost[w] * prov_lp[w] for w in inst.warehouses))
m_lp.addConstrs((ship_lp.sum("*", c) >= inst.demand[c] for c in inst.customers), name="demand")
m_lp.addConstrs((ship_lp.sum(w, "*") <= prov_lp[w] for w in inst.warehouses), name="capacity_used")
m_lp.addConstrs((prov_lp[w] <= inst.max_capacity[w] * open_lp[w] for w in inst.warehouses), name="capacity_built")
m_lp.optimize()

lp_obj = m_lp.ObjVal
lp_open = {w: open_lp[w].X for w in inst.warehouses}
print(f"relaxation total: {lp_obj:,.2f}     (integer answer was {mip_obj:,.0f})\n")
for w in inst.warehouses:
    print(f"  warehouse {w}: open = {lp_open[w]:.3f}")

relaxation total: 8,695.00     (integer answer was 9,880)

  warehouse 1: open = 0.812
  warehouse 2: open = 0.500
  warehouse 3: open = 0.625
  warehouse 4: open = 0.500
  warehouse 5: open = 0.000


Four sites partially open, and the total is lower. The relaxation opens exactly as much of each
warehouse as it needs to cover the shipments through it — a warehouse shipping 40 units of an 80-unit
capacity is "half open" and pays half its fixed cost. No real warehouse works like that, so this
answer cannot be built. But it is a **lower bound**: it does everything the integer model does with
fewer rules, so nothing feasible for the integer model can beat it.

In [11]:
gap = mip_obj - lp_obj
print(f"integer optimum   {mip_obj:9,.2f}")
print(f"LP relaxation     {lp_obj:9,.2f}")
print(f"integrality gap   {gap:9,.2f}   ({100 * gap / lp_obj:.1f}% of the bound)")
assert lp_obj <= mip_obj + tolerance.FEASIBILITY_ATOL, "a relaxation cannot cost more than what it relaxes"

integer optimum    9,880.00
LP relaxation      8,695.00
integrality gap    1,185.00   (13.6% of the bound)


## Branch and bound, by hand, for one level

The solver closed that gap. Here is what it did, once.

Pick a fractional site — the one nearest to 0.5, because it is the one the relaxation is least sure
about. Split the problem in two: one copy where that site is **forced closed**, one where it is
**forced open**. Each child is a relaxation with one more rule, so each child's cost is at least the
parent's. And the true optimum lives in one of them, so the *cheaper* child is a new, tighter bound.

**Predict:** will either child come back with every site at 0 or 1?

In [12]:
fractional = {w: v for w, v in lp_open.items() if 1e-9 < v < 1 - 1e-9}
branch_on = min(fractional, key=lambda w: abs(fractional[w] - 0.5))
print(f"fractional sites: {list(fractional)}   branching on warehouse {branch_on} (open = {lp_open[branch_on]:.3f})")

children = {}
for forced in (0, 1):
    mc = gp.Model(env=env)
    tolerance.apply(mc)
    mc.ModelSense = gp.GRB.MINIMIZE
    s = mc.addVars(inst.shipping.keys(), lb=0.0)
    o = mc.addVars(inst.warehouses, lb=0.0, ub=1.0)
    p = mc.addVars(inst.warehouses, lb=0.0)
    o[branch_on].LB = o[branch_on].UB = forced              # the one new rule
    mc.setObjective(gp.quicksum(inst.shipping[a] * s[a] for a in inst.shipping)
                    + gp.quicksum(inst.fixed_cost[w] * o[w] for w in inst.warehouses)
                    + gp.quicksum(inst.variable_cost[w] * p[w] for w in inst.warehouses))
    mc.addConstrs((s.sum("*", c) >= inst.demand[c] for c in inst.customers))
    mc.addConstrs((s.sum(w, "*") <= p[w] for w in inst.warehouses))
    mc.addConstrs((p[w] <= inst.max_capacity[w] * o[w] for w in inst.warehouses))
    mc.optimize()
    children[forced] = (mc.ObjVal, {w: o[w].X for w in inst.warehouses})
    integral = all(min(abs(v), abs(v - 1)) < 1e-9 for v in children[forced][1].values())
    print(f"\n  open[{branch_on}] = {forced}:  cost {mc.ObjVal:9,.2f}   integral: {integral}")
    print("     " + "  ".join(f"{w}:{v:.2f}" for w, v in children[forced][1].items()))

fractional sites: ['1', '2', '3', '4']   branching on warehouse 2 (open = 0.500)

  open[2] = 0:  cost  9,098.75   integral: False
     1:1.00  2:0.00  3:0.94  4:0.50  5:0.00

  open[2] = 1:  cost  9,310.00   integral: False
     1:0.31  2:1.00  3:0.62  4:0.50  5:0.00


Neither child is integral — the fraction just moved to a different site. That is normal, and it is why
the method is a *tree*: each child gets branched again, and again, until the leaves are whole. What
one level did buy is a better bound.

In [13]:
best_child = min(children[0][0], children[1][0])
print(f"bound before branching : {lp_obj:9,.2f}")
print(f"bound after one level  : {best_child:9,.2f}   (the cheaper child)")
print(f"integer optimum        : {mip_obj:9,.2f}")
print(f"gap remaining          : {mip_obj - best_child:9,.2f}   of the original {gap:,.2f}")
assert children[0][0] >= lp_obj - tolerance.FEASIBILITY_ATOL
assert children[1][0] >= lp_obj - tolerance.FEASIBILITY_ATOL
assert best_child <= mip_obj + tolerance.FEASIBILITY_ATOL, "the optimum must lie under the better child"

bound before branching :  8,695.00
bound after one level  :  9,098.75   (the cheaper child)
integer optimum        :  9,880.00
gap remaining          :    781.25   of the original 1,185.00


The solver keeps going: branch the cheaper child, then its cheaper child, pruning any node whose bound
already exceeds the best whole answer found so far. When no open node can beat the incumbent, the
incumbent is proven optimal. The `MIPGap 0` this library solves at means it does not stop early — the
bound and the incumbent meet exactly, and that is what makes comparing two integer solves to `1e-9`
a meaningful check rather than a coincidence.

---

# Now the streamlined version

The integer model, the relaxation, and the children differ only in the type of `open` and in what is
pinned, so they are one function with two arguments. `orteach.facility_location` takes the tables as
an argument and never reads a file.

In [14]:
from orteach import facility_location as fl
from orteach.tolerance import AGREEMENT_RTOL, rel_diff

pkg_mip = fl.solve(inst, env=env)
pkg_lp  = fl.solve(inst, relax=True, env=env)
pkg_kid = {v: fl.solve(inst, relax=True, fix_open={branch_on: v}, env=env) for v in (0, 1)}

print(f"{'MIP':16} {pkg_mip.objective:9,.2f}   opens {pkg_mip.opened()}   proven bound {pkg_mip.bound:,.2f}")
print(f"{'LP relaxation':16} {pkg_lp.objective:9,.2f}   most fractional: {fl.most_fractional(pkg_lp)}")
print(f"{'gap':16} {fl.integrality_gap(pkg_mip, pkg_lp):9,.2f}")

MIP               9,880.00   opens ['1', '3', '4']   proven bound 9,880.00
LP relaxation     8,695.00   most fractional: 2
gap               1,185.00


## The agreement assertion

The hand-built MIP, relaxation and both children against the package — objectives, every opening,
every shipment, every provisioned capacity. The MIP was solved to a zero gap on both sides, so the
comparison is between two proven optima and not two places a solver happened to stop.

In [15]:
checks = [("MIP objective", mip_obj, pkg_mip.objective),
          ("LP objective", lp_obj, pkg_lp.objective),
          ("child open=0", children[0][0], pkg_kid[0].objective),
          ("child open=1", children[1][0], pkg_kid[1].objective),
          ("MIP proven bound", mip_obj, pkg_mip.bound)]
for w in inst.warehouses:
    checks.append((f"MIP open[{w}]", mip_open[w], pkg_mip.open[w]))
    checks.append((f"LP open[{w}]", lp_open[w], pkg_lp.open[w]))
    checks.append((f"provisioned[{w}]", mip_prov[w], pkg_mip.provisioned[w]))
for a in inst.shipping:
    checks.append((f"ship{a}", mip_ship[a], pkg_mip.ship[a]))

worst = max(rel_diff(h, p) for _, h, p in checks)
print(f"{len(checks)} comparisons, worst relative difference {worst:.2e}")
assert worst < AGREEMENT_RTOL, f"notebook and package disagree by {worst:.2e}"
print(f"notebook and package agree to {worst:.1e}")

45 comparisons, worst relative difference 0.00e+00
notebook and package agree to 0.0e+00


---

## Where to take this next

- Double every fixed cost. Does the relaxation's gap grow or shrink, and does the set of opened sites
  change? Fixed costs are what integrality is *about*; watch what they do to the bound.
- Branch the cheaper child yourself, one more level. How many levels until a leaf is integral, and
  what does the bound look like when it gets there?
- Drop `provisioned` and write `ship.sum(w, "*") <= max_capacity[w] * open[w]` directly. Confirm the
  answer does not change, then decide which formulation you would teach and why.